# Frequency Model Development

## General Insurance Capital & Risk Modelling

### Modelling Objective

The purpose of this stage is to develop and validate a statistical model for
claim frequency across the insurance portfolio.

Claim frequency represents the expected number of claims per unit of exposure.
Because policy records have different exposure periods, exposure will be
incorporated through a log-exposure offset rather than treated as an ordinary
explanatory variable.

The modelling process will:

1. Establish a baseline Poisson frequency model.
2. Assess whether the observed claim counts exhibit overdispersion relative
   to the Poisson assumption.
3. Evaluate alternative count models where supported by the data.
4. Introduce relevant policyholder, vehicle and geographical risk factors.
5. Compare candidate models using statistical diagnostics and predictive
   performance.
6. Select and validate a final frequency model suitable for subsequent
   aggregate-loss and capital-risk modelling.

The objective is not simply to maximise predictive performance. The selected
model should also be statistically defensible, interpretable and appropriate
for the business purpose of estimating insurance loss risk.

## Modelling Framework

**Response variable**

- `ClaimNb` — observed number of claims for each policy record.

**Exposure**

- `Exposure` — policy exposure measured in policy-years.
- `log(Exposure)` will be used as an offset in the count model.

**Candidate risk factors**

Based on the data-quality review and exploratory analysis, candidate predictors
include:

- `DrivAge`
- `BonusMalus`
- `VehAge`
- `VehPower`
- `VehBrand`
- `VehGas`
- `Area`
- `Density`
- `Region`

Predictor inclusion in the final model will depend on statistical evidence,
model diagnostics, stability, interpretability and business relevance.

**Candidate model families**

- Poisson GLM
- Negative Binomial GLM, if overdispersion provides sufficient justification

**Model assessment**

Candidate models will be evaluated using appropriate measures including:

- dispersion diagnostics
- log-likelihood
- AIC
- deviance
- coefficient significance and stability
- observed versus predicted claim frequency
- out-of-sample predictive performance

Model complexity will only be introduced where it produces a justified
improvement in the representation of portfolio claim risk.

## 1. Data Preparation

The validated frequency dataset is loaded and prepared for statistical
modelling. The original source data are retained unchanged; modelling-specific
transformations are created within this notebook.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Load frequency dataset
freq = pd.read_csv("../data/processed/freMTPL2freq.csv")

print(f"Rows: {len(freq):,}")
print(f"Columns: {freq.shape[1]}")
freq.head()

Rows: 677,991
Columns: 12


,IDpol,ClaimNb,Exposure,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Area,Density,Region
0,1.0,0,0.10,5,0,55,50,B12,Regular,D,1217,Rhone-Alpes
1,3.0,0,0.77,5,0,55,50,B12,Regular,D,1217,Rhone-Alpes
2,5.0,0,0.75,6,2,52,50,B12,Diesel,B,54,Picardie
3,10.0,0,0.09,7,0,46,50,B12,Diesel,B,76,Aquitaine
4,11.0,0,0.84,7,0,46,50,B12,Diesel,B,76,Aquitaine


## 2. Modelling Variable Verification

Before fitting the claim frequency model, the response variable, exposure measure and candidate risk factors are verified to ensure that the modelling dataset remains consistent with the validated portfolio data.

In [3]:
# Verify key modelling variables

model_vars = [
    "ClaimNb",
    "Exposure",
    "VehPower",
    "VehAge",
    "DrivAge",
    "BonusMalus",
    "VehBrand",
    "VehGas",
    "Area",
    "Density",
    "Region"
]

verification = pd.DataFrame({
    "DataType": freq[model_vars].dtypes.astype(str),
    "Missing": freq[model_vars].isna().sum(),
    "UniqueValues": freq[model_vars].nunique()
})

verification

,DataType,Missing,UniqueValues
ClaimNb,int64,0,11
Exposure,float64,0,181
VehPower,int64,0,12
VehAge,int64,0,78
DrivAge,int64,0,83
BonusMalus,int64,0,115
VehBrand,object,0,11
VehGas,object,0,2
Area,object,0,6
Density,int64,0,1607


## 3. Baseline Poisson Frequency Model

A Poisson Generalised Linear Model (GLM) is used as the baseline claim frequency model.

The response variable is the number of reported claims (`ClaimNb`), with the logarithm of policy exposure included as an offset. This allows expected claim counts to vary appropriately with the period of risk exposure.

The initial model uses a focused set of established portfolio risk characteristics:

- Driver age
- Vehicle age
- Vehicle power
- Bonus-Malus score
- Vehicle fuel type
- Area

This baseline provides an interpretable actuarial benchmark before considering alternative specifications or additional risk factors.

In [4]:
# Create modelling dataset

model_data = freq[
    [
        "ClaimNb",
        "Exposure",
        "DrivAge",
        "VehAge",
        "VehPower",
        "BonusMalus",
        "VehGas",
        "Area"
    ]
].copy()

# Verify exposure is strictly positive before taking logarithms
print(f"Observations: {len(model_data):,}")
print(f"Exposure <= 0: {(model_data['Exposure'] <= 0).sum():,}")
print(f"Total exposure: {model_data['Exposure'].sum():,.2f}")
print(f"Total claims: {model_data['ClaimNb'].sum():,}")

# Log exposure for Poisson GLM offset
model_data["LogExposure"] = np.log(model_data["Exposure"])

Observations: 677,991
Exposure <= 0: 0
Total exposure: 358,482.84
Total claims: 26,444


### 3.1 Baseline Model Specification

The baseline model assumes that claim counts follow a Poisson distribution with a log link function.

Policy exposure is incorporated through a log-exposure offset. Driver age, vehicle age, vehicle power and Bonus-Malus are initially treated as continuous rating factors, while fuel type and area are treated as categorical factors.

The purpose of this model is to establish an interpretable benchmark for claim frequency before model diagnostics and refinement.

In [5]:
# Specify and fit baseline Poisson GLM

baseline_formula = (
    "ClaimNb ~ DrivAge + VehAge + VehPower + "
    "BonusMalus + C(VehGas) + C(Area)"
)

poisson_baseline = smf.glm(
    formula=baseline_formula,
    data=model_data,
    family=sm.families.Poisson(),
    offset=model_data["LogExposure"]
).fit()

print(poisson_baseline.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               677991
Model:                            GLM   Df Residuals:                   677980
Model Family:                 Poisson   Df Model:                           10
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.0803e+05
Date:                Wed, 23 Sep 2026   Deviance:                   1.6530e+05
Time:                        18:06:03   Pearson chi2:                 1.22e+06
No. Iterations:                     7   Pseudo R-squ. (CS):           0.009569
Covariance Type:            nonrobust                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               -4.7233 

### 3.2 Baseline Model Diagnostics

The baseline Poisson model is assessed for goodness of fit and dispersion.

Under the Poisson assumption, the conditional variance of claim counts is expected to equal the conditional mean. A Pearson dispersion statistic materially above one may therefore indicate overdispersion and suggests that the baseline Poisson variance assumption requires further investigation.

The diagnostics below evaluate the model before any refinement or alternative specification is introduced.

In [6]:
# Baseline Poisson model diagnostics

pearson_chi2 = poisson_baseline.pearson_chi2
df_resid = poisson_baseline.df_resid
dispersion = pearson_chi2 / df_resid

print(f"Log-Likelihood: {poisson_baseline.llf:,.2f}")
print(f"Deviance: {poisson_baseline.deviance:,.2f}")
print(f"Pearson Chi-Square: {pearson_chi2:,.2f}")
print(f"Residual Degrees of Freedom: {df_resid:,}")
print(f"Pearson Dispersion Statistic: {dispersion:.4f}")
print(f"AIC: {poisson_baseline.aic:,.2f}")

Log-Likelihood: -108,032.97
Deviance: 165,297.80
Pearson Chi-Square: 1,215,206.17
Residual Degrees of Freedom: 677,980
Pearson Dispersion Statistic: 1.7924
AIC: 216,087.94


### 3.3 Overdispersion Assessment

The baseline Poisson model produces a Pearson dispersion statistic of **1.7924**.

A value close to one would be consistent with the Poisson mean-variance assumption. The observed value above one indicates overdispersion: claim-count variability is greater than that implied by the fitted Poisson model.

This does not invalidate the Poisson model as a benchmark, but it indicates that a more flexible frequency specification should be investigated before the model is used within the aggregate loss and capital simulation framework.

A Negative Binomial model will therefore be considered as an alternative because it allows the conditional variance of claim counts to exceed the conditional mean.

In [7]:
# Examine observed claim-count distribution

claim_distribution = (
    model_data["ClaimNb"]
    .value_counts()
    .sort_index()
    .rename_axis("ClaimCount")
    .to_frame("Policies")
)

claim_distribution["PolicyShare"] = (
    claim_distribution["Policies"] / len(model_data)
)

claim_distribution["PolicyShare"] = (
    claim_distribution["PolicyShare"] * 100
).round(4)

claim_distribution

,Policies,PolicyShare
ClaimCount,,
0,653047,96.3209
1,23571,3.4766
2,1298,0.1914
3,62,0.0091
4,5,0.0007
5,2,0.0003
6,1,0.0001
8,1,0.0001
9,1,0.0001


## 4. Negative Binomial Frequency Model

The baseline Poisson model identified material overdispersion, with a Pearson dispersion statistic of 1.7924.

The observed claim-count distribution also shows a highly concentrated zero-claim population alongside a small tail of policies experiencing multiple claims. This suggests that claim-count variability is not fully represented by the Poisson mean-variance assumption.

A Negative Binomial model is therefore fitted as an alternative frequency specification. The model retains the same risk factors and exposure treatment as the baseline Poisson model so that model performance can be compared on a consistent basis.

In [8]:
# Estimate Negative Binomial frequency model

import statsmodels.discrete.discrete_model as discrete

nb_formula = (
    "ClaimNb ~ DrivAge + VehAge + VehPower + "
    "BonusMalus + C(VehGas) + C(Area)"
)

negative_binomial = smf.negativebinomial(
    formula=nb_formula,
    data=model_data,
    offset=model_data["LogExposure"]
).fit(
    maxiter=100,
    disp=False
)

print(negative_binomial.summary())

c:\Users\utulu\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\discrete\discrete_model.py:3384: RuntimeWarning: overflow encountered in exp
  alpha = np.exp(params[-1])
c:\Users\utulu\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\discrete\discrete_model.py:3377: RuntimeWarning: invalid value encountered in subtract
  coeff = (gamma_ln(size+endog) - gamma_ln(endog+1) -
c:\Users\utulu\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\discrete\discrete_model.py:3379: RuntimeWarning: divide by zero encountered in log
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)
c:\Users\utulu\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\discrete\discrete_model.py:3379: RuntimeWarning: invalid value encountered in multiply
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)
c:\Users\utulu\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\discrete\discrete_model.py:3448: RuntimeWarn

                     NegativeBinomial Regression Results                      
Dep. Variable:                ClaimNb   No. Observations:               677991
Model:               NegativeBinomial   Df Residuals:                   677980
Method:                           MLE   Df Model:                           10
Date:                Wed, 23 Sep 2026   Pseudo R-squ.:                     nan
Time:                        18:50:23   Log-Likelihood:                    nan
converged:                      False   LL-Null:                   -1.1067e+05
Covariance Type:            nonrobust   LLR p-value:                       nan
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               -4.7233        nan        nan        nan         nan         nan
C(VehGas)[T.Regular]    -0.1834        nan        nan        nan         nan         nan
C(Area)[T.B]

### 4.1 Negative Binomial Dispersion Estimation

Direct maximum-likelihood estimation of the Negative Binomial model did not converge, producing an infinite dispersion parameter and undefined likelihood statistics.

Rather than relying on the failed specification, the Negative Binomial dispersion parameter is estimated from the baseline Poisson model using the relationship between squared Pearson residuals and fitted claim counts.

This provides a data-driven estimate of the additional variance beyond the Poisson assumption before refitting the frequency model using a Negative Binomial GLM.

In [9]:
# Estimate Negative Binomial alpha from Poisson residual behaviour

mu = poisson_baseline.fittedvalues
y = model_data["ClaimNb"]

numerator = ((y - mu) ** 2 - y).sum()
denominator = (mu ** 2).sum()

alpha_est = max(numerator / denominator, 0)

print(f"Estimated Negative Binomial alpha: {alpha_est:.6f}")

Estimated Negative Binomial alpha: 1.074198


### 4.2 Negative Binomial GLM

Using the empirically estimated dispersion parameter, a Negative Binomial GLM is fitted with the same explanatory variables and exposure offset as the baseline Poisson model.

Keeping the model structure unchanged provides a consistent comparison between the Poisson and Negative Binomial specifications while allowing the Negative Binomial model to accommodate additional claim-count variability.

In [10]:
# Fit Negative Binomial GLM using estimated alpha

nb_glm = smf.glm(
    formula=baseline_formula,
    data=model_data,
    family=sm.families.NegativeBinomial(alpha=alpha_est),
    offset=model_data["LogExposure"]
).fit()

print(nb_glm.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               677991
Model:                            GLM   Df Residuals:                   677980
Model Family:        NegativeBinomial   Df Model:                           10
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.0769e+05
Date:                Wed, 23 Sep 2026   Deviance:                   1.4358e+05
Time:                        20:21:54   Pearson chi2:                 1.18e+06
No. Iterations:                     7   Pseudo R-squ. (CS):           0.008916
Covariance Type:            nonrobust                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               -4.7444 

### 4.3 Poisson vs Negative Binomial Model Comparison

The Negative Binomial model is compared with the baseline Poisson model using log-likelihood, AIC, deviance and Pearson dispersion.

The objective is to determine whether explicitly allowing for overdispersion provides a materially better representation of claim-frequency risk.

In [13]:
# Compare Poisson and Negative Binomial frequency models

comparison = pd.DataFrame({
    "Model": ["Poisson GLM", "Negative Binomial GLM"],
    "LogLikelihood": [
        poisson_baseline.llf,
        nb_glm.llf
    ],
    "AIC": [
        poisson_baseline.aic,
        nb_glm.aic
    ],
    "Deviance": [
        poisson_baseline.deviance,
        nb_glm.deviance
    ],
    "PearsonDispersion": [
        poisson_baseline.pearson_chi2 / poisson_baseline.df_resid,
        nb_glm.pearson_chi2 / nb_glm.df_resid
    ]
})

comparison.round({
    "LogLikelihood": 2,
    "AIC": 2,
    "Deviance": 2,
    "PearsonDispersion": 4
})

,Model,LogLikelihood,AIC,Deviance,PearsonDispersion
0,Poisson GLM,-108032.97,216087.94,165297.80,1.7924
1,Negative Binomial GLM,-107689.83,215401.66,143575.54,1.7446


### Interpretation

The baseline Poisson model exhibited material overdispersion, with a Pearson dispersion statistic of 1.7924, indicating greater claim-count variability than implied by the Poisson variance assumption.

Introducing a Negative Binomial variance structure improved model fit. The Negative Binomial GLM produced a higher log-likelihood, lower AIC and substantially lower deviance than the Poisson model.

However, the Pearson dispersion statistic remained elevated at 1.7446. The Negative Binomial specification therefore reduced, but did not fully eliminate, the residual overdispersion in claim frequency.

For the subsequent capital modelling framework, the Negative Binomial model is retained as the preferred frequency specification because it provides a better representation of claim-count variability than the baseline Poisson model. The remaining dispersion will be recognised as a model limitation when interpreting simulated tail losses and capital estimates.

## 5. Frequency Model Outputs for Capital Modelling

The fitted Negative Binomial model is now translated into portfolio-level expected claim frequencies.

These outputs provide the frequency component required for the later aggregate-loss simulation, where claim frequency will be combined with a severity model to generate annual insurance losses.

The first step is to calculate the fitted expected claim count for each policy exposure and validate the model at portfolio level.

In [14]:
# Generate fitted expected claim counts from the selected frequency model

model_data["ExpectedClaims_NB"] = nb_glm.predict(
    model_data,
    offset=model_data["LogExposure"]
)

observed_claims = model_data["ClaimNb"].sum()
expected_claims = model_data["ExpectedClaims_NB"].sum()
total_exposure = model_data["Exposure"].sum()

observed_frequency = observed_claims / total_exposure
expected_frequency = expected_claims / total_exposure

print("PORTFOLIO FREQUENCY MODEL VALIDATION")
print(f"Observed claims:           {observed_claims:,.0f}")
print(f"Expected claims (NB):      {expected_claims:,.2f}")
print(f"Total exposure:            {total_exposure:,.2f}")
print(f"Observed claim frequency:  {observed_frequency:.4f}")
print(f"Expected claim frequency:  {expected_frequency:.4f}")
print(f"Observed / Expected ratio: {observed_claims / expected_claims:.4f}")

PORTFOLIO FREQUENCY MODEL VALIDATION
Observed claims:           26,444
Expected claims (NB):      26,578.40
Total exposure:            358,482.84
Observed claim frequency:  0.0738
Expected claim frequency:  0.0741
Observed / Expected ratio: 0.9949


### Interpretation

The selected Negative Binomial frequency model reproduces the portfolio-level claim experience closely.

Observed claim frequency is 0.0738 claims per unit of exposure, compared with a modelled frequency of 0.0741. The observed-to-expected claim-count ratio is 0.9949, indicating that aggregate expected claims are approximately 0.5% higher than the observed total.

This close portfolio-level calibration provides a reasonable basis for using the fitted frequency model as the frequency component of the subsequent aggregate-loss simulation. This validation concerns aggregate calibration and does not imply that all individual risk segments are equally well calibrated.

### 5.1 Segment-Level Frequency Calibration

Portfolio-level calibration can conceal differences between risk groups. Observed and expected claim frequencies are therefore compared across major rating characteristics to assess whether the selected model represents heterogeneous portfolio risks reasonably well.

In [15]:
# Validate observed versus expected claim frequency by Area

area_validation = (
    model_data
    .groupby("Area", observed=True)
    .agg(
        Exposure=("Exposure", "sum"),
        ObservedClaims=("ClaimNb", "sum"),
        ExpectedClaims=("ExpectedClaims_NB", "sum")
    )
)

area_validation["ObservedFrequency"] = (
    area_validation["ObservedClaims"] / area_validation["Exposure"]
)

area_validation["ExpectedFrequency"] = (
    area_validation["ExpectedClaims"] / area_validation["Exposure"]
)

area_validation["OE_Ratio"] = (
    area_validation["ObservedClaims"] / area_validation["ExpectedClaims"]
)

area_validation.round({
    "Exposure": 2,
    "ExpectedClaims": 2,
    "ObservedFrequency": 4,
    "ExpectedFrequency": 4,
    "OE_Ratio": 4
})

,Exposure,ObservedClaims,ExpectedClaims,ObservedFrequency,ExpectedFrequency,OE_Ratio
Area,,,,,,
A,61966.14,3364,3367.16,0.0543,0.0543,0.9991
B,43010.75,2633,2638.04,0.0612,0.0613,0.9981
C,104443.16,7093,7120.57,0.0679,0.0682,0.9961
D,77116.92,6458,6504.34,0.0837,0.0843,0.9929
E,63817.10,6122,6165.51,0.0959,0.0966,0.9929
F,8128.75,774,782.79,0.0952,0.0963,0.9888


#### Interpretation

The Negative Binomial model demonstrates strong calibration across geographic risk areas. Observed-to-expected ratios range from 0.9888 to 0.9991, with modelled claim frequencies closely tracking observed frequencies across all six Area categories.

The model also preserves the underlying geographic risk gradient, with expected claim frequency increasing materially from lower-risk Area A to the higher-frequency Area E and F segments.

This provides evidence that the strong portfolio-level calibration is not being achieved by offsetting substantial miscalibration between geographic risk groups.

### 5.2 Calibration by Bonus-Malus Risk Band

Observed and expected claim frequencies are compared across Bonus-Malus bands to assess whether the selected frequency model captures the strong risk differentiation identified during exploratory analysis.

In [16]:
# Create Bonus-Malus bands for model validation

model_data["BonusMalusBand"] = pd.cut(
    model_data["BonusMalus"],
    bins=[49, 59, 79, 99, 119, 149, float("inf")],
    labels=["50-59", "60-79", "80-99", "100-119", "120-149", "150+"]
)

bonus_validation = (
    model_data
    .groupby("BonusMalusBand", observed=True)
    .agg(
        Exposure=("Exposure", "sum"),
        ObservedClaims=("ClaimNb", "sum"),
        ExpectedClaims=("ExpectedClaims_NB", "sum")
    )
)

bonus_validation["ObservedFrequency"] = (
    bonus_validation["ObservedClaims"] / bonus_validation["Exposure"]
)

bonus_validation["ExpectedFrequency"] = (
    bonus_validation["ExpectedClaims"] / bonus_validation["Exposure"]
)

bonus_validation["OE_Ratio"] = (
    bonus_validation["ObservedClaims"] / bonus_validation["ExpectedClaims"]
)

bonus_validation.round({
    "Exposure": 2,
    "ExpectedClaims": 2,
    "ObservedFrequency": 4,
    "ExpectedFrequency": 4,
    "OE_Ratio": 4
})

,Exposure,ObservedClaims,ExpectedClaims,ObservedFrequency,ExpectedFrequency,OE_Ratio
BonusMalusBand,,,,,,
50-59,265051.74,14626,15233.72,0.0552,0.0575,0.9601
60-79,53803.92,5750,4707.42,0.1069,0.0875,1.2215
80-99,29814.98,3492,4223.02,0.1171,0.1416,0.8269
100-119,8806.77,2119,1911.70,0.2406,0.2171,1.1084
120-149,910.30,403,391.78,0.4427,0.4304,1.0286
150+,95.13,54,110.74,0.5677,1.1642,0.4876


#### Interpretation

Calibration across Bonus-Malus bands is less uniform than the portfolio-level and Area-level results.

The model preserves the broad increase in claim frequency associated with higher Bonus-Malus values, but material differences between observed and expected experience remain in some bands. In particular, the model underpredicts claims in the 60–79 band and overpredicts claims in the 80–99 band.

The 150+ band shows the largest relative difference, with an observed-to-expected ratio of 0.4876. However, this segment represents only 95.13 units of exposure, making its observed experience considerably less credible than that of the larger bands.

These results indicate that the selected Negative Binomial model provides a reasonable portfolio-level frequency model for the capital-modelling exercise, while retaining segment-level calibration limitations that should be recognised when interpreting results for individual risk groups.

### 5.3 Frequency Model Output

The selected Negative Binomial model provides the frequency component for the subsequent aggregate-loss modelling framework.

Policy-level fitted expected claim counts are retained so that expected claim volume can be aggregated consistently across portfolio segments during the capital simulation.

In [18]:
# Save frequency-model outputs for subsequent modelling

frequency_output = freq[
    [
        "IDpol",
        "Exposure",
        "Area",
        "VehGas",
        "DrivAge",
        "VehAge",
        "VehPower",
        "BonusMalus"
    ]
].copy()

# Add fitted expected claims from the selected Negative Binomial model
frequency_output["ExpectedClaims_NB"] = nb_glm.predict(
    model_data,
    offset=model_data["LogExposure"]
)

frequency_output.to_csv(
    "../outputs/tables/frequency_model_output.csv",
    index=False
)

print(f"Rows exported: {len(frequency_output):,}")
print(
    f"Total expected claims exported: "
    f"{frequency_output['ExpectedClaims_NB'].sum():,.2f}"
)

Rows exported: 677,991
Total expected claims exported: 26,578.40
